# Equality Classifier — 500 k Text Pipeline

**Three categories:** `PRO_EQUALITY` | `ANTI_EQUALITY` | `NEUTRAL`  
**Three free models:** Groq (Llama-3.3-70B) · Gemini Flash 2.0 Lite · OpenRouter (Llama-3.3-70B:free)  
**Optimised for Google Colab + 500 k texts** — async requests, parquet checkpoints, resume-safe.

---
## A — Setup

In [ ]:
!pip install google-generativeai openai aiohttp pandas numpy scipy matplotlib seaborn tqdm pyarrow nest_asyncio scikit-learn --quiet

In [ ]:
import os, json, re, time, math, hashlib, asyncio, html, unicodedata
from pathlib import Path
from collections import Counter
import nest_asyncio; nest_asyncio.apply()   # needed for top-level await in Colab/Jupyter
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import aiohttp
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import warnings; warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Pandas {pd.__version__} | Colab={IN_COLAB} | Async ready')


In [ ]:
# ── API keys (fill in before running) ─────────────────────────────────
GROQ_API_KEY       = ''   # console.groq.com          — free tier
OPENROUTER_API_KEY = ''   # openrouter.ai             — free models
GEMINI_API_KEY     = ''   # aistudio.google.com       — free tier

# ── Models ──────────────────────────────────────────────────────────────
GROQ_MODEL       = 'llama-3.3-70b-versatile'              # free, fast
GEMINI_MODEL     = 'gemini-2.0-flash-lite'                # free
OR_MODEL         = 'meta-llama/llama-3.3-70b-instruct:free'  # free
# Alt OpenRouter: 'qwen/qwen3-235b-a22b:free'  (add /no_think prefix for Qwen)

# ── Paths (adjust to your Drive layout) ─────────────────────────────────
INPUT_PATH     = '/content/drive/MyDrive/RA/RA/IncomeAndWealth.csv'
CKPT_DIR       = '/content/drive/MyDrive/RA/RA/output/equality/'
PRIMARY_CKPT   = CKPT_DIR + 'primary.parquet'
CROSSVAL_PATH  = CKPT_DIR + 'crossval.parquet'
FINAL_PATH     = CKPT_DIR + 'results_final.parquet'

# ── Classification ───────────────────────────────────────────────────────
CATEGORIES     = ['PRO_EQUALITY', 'ANTI_EQUALITY', 'NEUTRAL']
BATCH_SIZE     = 20      # texts per API call
MAX_CHARS      = 1200    # truncate longer texts (keeps first ~300 words)
SAVE_EVERY     = 5_000   # checkpoint every N texts
MAX_CONCURRENT = 5       # async workers (stay within 30 req/min Groq limit)
SLEEP_GROQ     = 2.1     # seconds between batch-groups (30 req/min → 2s gap)
SLEEP_GEMINI   = 1.0
SLEEP_OR       = 4.0     # OpenRouter free tier is slower

# ── Cross-validation ─────────────────────────────────────────────────────
N_PER_CAT = 500          # sample size per category for cross-val

print('Config OK')


In [ ]:
# Mount Google Drive (skip if running locally)
if IN_COLAB:
    drive.mount('/content/drive')

Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
print('Directories ready:', CKPT_DIR)


---
## B — Data Preparation

In [ ]:
# Load raw dataset
print('Loading dataset...')
df = pd.read_csv(INPUT_PATH)
df['id'] = df['id'].astype(str)
print(f'Raw records: {len(df):,}')
print(df[['id', 'text']].head(3))


In [ ]:
def clean_text(text: str, max_chars: int = MAX_CHARS):
    """Normalise, strip HTML/URLs/control-chars, validate length, truncate."""
    if not isinstance(text, str):
        return None
    text = html.unescape(text)                          # &amp; → &
    text = re.sub(r'https?://\S+', '', text)           # drop URLs
    text = unicodedata.normalize('NFKC', text)          # unicode normalise
    text = re.sub(r'[\x00-\x1f\x7f-\x9f]', ' ', text)  # control chars
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) < 50:
        return None                                      # too short
    # Truncate at sentence boundary — preserves first ~300 words
    if len(text) > max_chars:
        cut = text[:max_chars]
        last = max(cut.rfind('. '), cut.rfind('! '), cut.rfind('? '))
        text = cut[:last + 1] if last > max_chars // 2 else cut + '…'
    return text

df['text_clean'] = df['text'].apply(clean_text)
n_invalid = df['text_clean'].isna().sum()
df = df.dropna(subset=['text_clean']).copy()
print(f'After cleaning: {len(df):,} texts  ({n_invalid:,} removed — too short / invalid)')


In [ ]:
# ── Deduplication ──────────────────────────────────────────────────────
# 1. Duplicate IDs
n0 = len(df)
df = df.drop_duplicates(subset='id')
print(f'Duplicate IDs removed:      {n0 - len(df):,}')

# 2. Identical text content (hash-based)
df['_hash'] = df['text_clean'].apply(lambda t: hashlib.sha256(t.encode()).hexdigest())
n1 = len(df)
df = df.drop_duplicates(subset='_hash').drop(columns='_hash')
print(f'Exact-content duplicates:   {n1 - len(df):,}')
print(f'\nFinal unique texts:         {len(df):,}')

# ── Text-length stats & throughput estimate ──────────────────────────────
lens = df['text_clean'].str.len()
print(f'\nText length after truncation (chars):')
print(f'  Mean {lens.mean():.0f}  Median {lens.median():.0f}  P95 {lens.quantile(0.95):.0f}  Max {lens.max():.0f}')

avg_tokens_per_batch = lens.mean() / 4 * BATCH_SIZE   # ~4 chars/token
# Groq free: 30 req/min AND 6000 tok/min — binding is tokens
batches_per_min = min(30, 6_000 / (avg_tokens_per_batch + BATCH_SIZE * 20))
texts_per_hr = batches_per_min * BATCH_SIZE * 60
print(f'\nGroq free-tier throughput estimate:')
print(f'  {batches_per_min:.1f} batches/min → {texts_per_hr:,.0f} texts/hour')
print(f'  ETA for {len(df):,} texts: ~{len(df)/texts_per_hr:.1f} hours (across sessions with checkpoint)')
print(f'  Tip: Gemini free-tier allows ~1 500 req/day → consider for primary if quota allows.')


---
## C — Prompt & Utilities

In [ ]:
EQUALITY_PROMPT = """\
Classify each news text about income/wealth inequality into exactly one category:

PRO_EQUALITY — The text supports or implies reducing inequality. Indicators:
  * Advocates redistribution, progressive taxation, or social spending
  * Frames inequality as structural, systemic, or policy-correctable
  * Calls for equal access to education, healthcare, or opportunities
  * Criticises economic elites or defends government intervention

ANTI_EQUALITY — The text defends or justifies inequality. Indicators:
  * Argues outcomes reflect individual merit, effort, or talent
  * Frames inequality as natural, efficient, or a fair market outcome
  * Opposes redistribution as harmful to incentives or growth
  * Suggests wealth differences are deserved

NEUTRAL — No clear normative stance: factual statistics, both sides presented
  equally, or topic unrelated to equality norms.

Return ONLY a JSON array, same order as inputs:
[{\"label\": \"PRO_EQUALITY\"|\"ANTI_EQUALITY\"|\"NEUTRAL\", \"conf\": 0.0-1.0}, ...]

conf = confidence (0.5 = uncertain, 1.0 = very certain). One label per text, no skipping.

Texts:
"""

VALID_LABELS = {'PRO_EQUALITY', 'ANTI_EQUALITY', 'NEUTRAL'}
print('Prompt length:', len(EQUALITY_PROMPT), 'chars')


In [ ]:
def parse_response(raw: str, expected_n: int) -> list:
    """Extract and validate JSON array from any LLM response."""
    # Strip thinking blocks (Qwen3, DeepSeek, o-series)
    raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    # Strip markdown fences
    raw = re.sub(r'```(?:json)?\s*', '', raw).strip().rstrip('`')
    # Extract JSON array
    m = re.search(r'\[\s*\{.*?\}\s*\]', raw, re.DOTALL)
    if not m:
        raise ValueError(f'No JSON array found. Got: {repr(raw[:300])}')
    items = json.loads(m.group(0))
    if len(items) != expected_n:
        raise ValueError(f'Expected {expected_n} items, got {len(items)}')
    for it in items:
        if it.get('label') not in VALID_LABELS:
            it['label'] = 'NEUTRAL'
            it['conf']  = 0.5
        it['conf'] = float(it.get('conf', 0.5))
    return items

print('parse_response OK')


In [ ]:
# ── Checkpoint helpers ──────────────────────────────────────────────────
def load_checkpoint(path: str) -> tuple:
    """Return (DataFrame, set_of_done_ids). Returns empty objects if no file."""
    p = Path(path)
    if p.exists():
        done = pd.read_parquet(p)
        done_ids = set(done['id'].tolist())
        print(f'Checkpoint loaded: {len(done):,} texts already classified')
        return done, done_ids
    print('No checkpoint found — starting fresh.')
    return pd.DataFrame(), set()

def save_checkpoint(new_records: list, path: str):
    """Append new records to parquet checkpoint (idempotent)."""
    if not new_records:
        return
    p = Path(path)
    new_df = pd.DataFrame(new_records)
    if p.exists():
        combined = pd.concat([pd.read_parquet(p), new_df], ignore_index=True)
        combined = combined.drop_duplicates(subset='id', keep='last')
    else:
        combined = new_df
    combined.to_parquet(p, index=False)
    print(f'  Checkpoint saved: {len(combined):,} total records → {path}')

print('Checkpoint helpers OK')


---
## D — Primary Classification (all 500 k texts)

> **Default primary model: Groq / Llama-3.3-70B** (async, free, generous token budget).  
> Switch `PRIMARY = 'gemini'` below to use Gemini Flash Lite instead.  
> The checkpoint system makes any multi-day run fully resumable.

In [ ]:
# ── Async Groq classifier ────────────────────────────────────────────────
async def _groq_batch(session, sem, batch_texts, batch_ids, prompt):
    url     = 'https://api.groq.com/openai/v1/chat/completions'
    headers = {'Authorization': f'Bearer {GROQ_API_KEY}', 'Content-Type': 'application/json'}
    payload = {
        'model': GROQ_MODEL,
        'messages': [{'role': 'user', 'content': prompt + json.dumps(batch_texts)}],
        'temperature': 0,
        'max_tokens': len(batch_texts) * 30 + 50,  # ~30 output tokens/text
    }
    async with sem:
        for attempt in range(4):
            try:
                async with session.post(url, headers=headers, json=payload,
                                        timeout=aiohttp.ClientTimeout(total=90)) as r:
                    if r.status == 429:
                        await asyncio.sleep(min(2 ** attempt * 10, 120))
                        continue
                    if r.status >= 400:
                        raise ValueError(f'HTTP {r.status}: {await r.text()}')
                    data    = await r.json()
                    content = data['choices'][0]['message']['content']
                    items   = parse_response(content, len(batch_ids))
                    return [{'id': bid, **it} for bid, it in zip(batch_ids, items)]
            except Exception as e:
                if attempt < 3:
                    await asyncio.sleep(2 ** attempt)
    # All retries failed
    return [{'id': bid, 'label': 'ERROR', 'conf': 0.0} for bid in batch_ids]


async def run_groq_async(texts, ids, batch_size=BATCH_SIZE,
                          max_concurrent=MAX_CONCURRENT, sleep=SLEEP_GROQ):
    """Async Groq classification — saturates 30 req/min free-tier limit."""
    batches = [(texts[i:i+batch_size], ids[i:i+batch_size])
               for i in range(0, len(texts), batch_size)]
    sem     = asyncio.Semaphore(max_concurrent)
    results = []
    conn    = aiohttp.TCPConnector(limit=max_concurrent + 4)
    async with aiohttp.ClientSession(connector=conn) as session:
        for i in tqdm(range(0, len(batches), max_concurrent),
                       desc='Groq/Llama-3.3-70B', unit='chunk'):
            chunk = batches[i: i + max_concurrent]
            tasks = [_groq_batch(session, sem, bt, bi, EQUALITY_PROMPT)
                     for bt, bi in chunk]
            for res in await asyncio.gather(*tasks):
                results.extend(res)
            await asyncio.sleep(sleep)
    return pd.DataFrame(results)

print('Async Groq classifier ready')


In [ ]:
# ── Sync Gemini classifier (alternative primary) ─────────────────────────
import google.generativeai as genai

def run_gemini(texts, ids, batch_size=BATCH_SIZE, sleep=SLEEP_GEMINI, desc='Gemini'):
    genai.configure(api_key=GEMINI_API_KEY)
    model   = genai.GenerativeModel(GEMINI_MODEL)
    results = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        bt, bi = texts[i:i+batch_size], ids[i:i+batch_size]
        for attempt in range(4):
            try:
                resp  = model.generate_content(EQUALITY_PROMPT + json.dumps(bt))
                items = parse_response(resp.text, len(bi))
                for bid, it in zip(bi, items):
                    results.append({'id': bid, **it})
                break
            except Exception as e:
                wait = 60 if '429' in str(e) else 2 ** attempt
                print(f'  Retry {attempt+1}: {str(e)[:80]}')
                time.sleep(wait)
        time.sleep(sleep)
    return pd.DataFrame(results)

print('Sync Gemini classifier ready')


In [ ]:
# ── Run primary classification with checkpoint ────────────────────────────
# Set PRIMARY = 'groq' or 'gemini'
PRIMARY = 'groq'

done_df, done_ids = load_checkpoint(PRIMARY_CKPT)
pending = df[~df['id'].isin(done_ids)].copy().reset_index(drop=True)
print(f'Total: {len(df):,}  |  Done: {len(done_ids):,}  |  Pending: {len(pending):,}')

if len(pending) > 0:
    tx_all = pending['text_clean'].tolist()
    id_all = pending['id'].tolist()
    n_chunks = math.ceil(len(tx_all) / SAVE_EVERY)

    for ci in range(n_chunks):
        s, e = ci * SAVE_EVERY, min((ci + 1) * SAVE_EVERY, len(tx_all))
        print(f'\n--- Chunk {ci+1}/{n_chunks}: texts {s:,}\u2013{e:,} ---')
        if PRIMARY == 'groq':
            chunk_df = await run_groq_async(tx_all[s:e], id_all[s:e])
        else:
            chunk_df = run_gemini(tx_all[s:e], id_all[s:e])

        errors = (chunk_df['label'] == 'ERROR').sum()
        dist   = chunk_df['label'].value_counts().to_dict()
        print(f'  Errors: {errors}  |  Distribution: {dist}')
        save_checkpoint(chunk_df.to_dict('records'), PRIMARY_CKPT)

print('\nPrimary classification complete.')


In [ ]:
# ── Load & summarise primary results ─────────────────────────────────────
results = pd.read_parquet(PRIMARY_CKPT)
results['id'] = results['id'].astype(str)
print(f'Primary results: {len(results):,} texts')
print()

# Distribution
dist = results['label'].value_counts()
pct  = results['label'].value_counts(normalize=True).mul(100)
print('Label distribution:')
for cat in CATEGORIES:
    n = dist.get(cat, 0)
    p = pct.get(cat, 0)
    print(f'  {cat:<18} {n:>8,}  ({p:.1f}%)')

n_err = (results['label'] == 'ERROR').sum()
if n_err:
    print(f'\n  ERROR (API failure): {n_err:,} — consider re-running pending texts')

print(f'\nMean confidence: {results["conf"].mean():.3f}')
print(f'Low-conf texts (conf < 0.6): {(results["conf"] < 0.6).sum():,}')


---
## E — Cross-Validation (stratified sample, 3 models)

We select up to **500 texts per category** from the primary results (highest confidence),
then classify them with **Gemini** and **OpenRouter** to measure inter-model agreement.

In [ ]:
# ── Build cross-val sample ───────────────────────────────────────────────
# Merge primary labels with cleaned texts
merged = results.merge(df[['id', 'text_clean']].rename(columns={'text_clean': 'text'}),
                        on='id', how='inner')
print(f'Texts with primary label AND original text: {len(merged):,}')

sample_ids = set()
for cat in CATEGORIES:
    pool = merged[(merged['label'] == cat) & (merged['label'] != 'ERROR')]
    top  = pool.nlargest(N_PER_CAT, 'conf')['id'].tolist()
    sample_ids.update(top)
    print(f'  {cat}: {len(top)} texts added')

sample = merged[merged['id'].isin(sample_ids)].copy().reset_index(drop=True)
print(f'\nCross-val sample (deduplicated): {len(sample)} texts')
print(sample['label'].value_counts().to_string())

cv_texts = sample['text'].tolist()
cv_ids   = sample['id'].tolist()


In [ ]:
# ── Model 2: Gemini cross-val ────────────────────────────────────────────
df_gemini = run_gemini(cv_texts, cv_ids, desc=f'Gemini/{GEMINI_MODEL}')
df_gemini = df_gemini.rename(columns={'label': 'label_gemini', 'conf': 'conf_gemini'})
print(f'Gemini: {len(df_gemini)} texts')
print(df_gemini['label_gemini'].value_counts().to_dict())


In [ ]:
# ── Model 3: OpenRouter cross-val ────────────────────────────────────────
from openai import OpenAI as ORClient

def run_openrouter(texts, ids, model=OR_MODEL,
                    batch_size=BATCH_SIZE, sleep=SLEEP_OR):
    client = ORClient(base_url='https://openrouter.ai/api/v1',
                      api_key=OPENROUTER_API_KEY)
    # Qwen3 needs /no_think to disable extended reasoning
    prefix  = '/no_think\n' if 'qwen' in model.lower() else ''
    results = []
    for i in tqdm(range(0, len(texts), batch_size),
                   desc=f'OpenRouter/{model.split("/")[-1][:20]}'):
        bt, bi = texts[i:i+batch_size], ids[i:i+batch_size]
        for attempt in range(4):
            try:
                resp  = client.chat.completions.create(
                    model=model,
                    messages=[{'role': 'user',
                               'content': prefix + EQUALITY_PROMPT + json.dumps(bt)}],
                    temperature=0,
                )
                items = parse_response(resp.choices[0].message.content, len(bi))
                for bid, it in zip(bi, items):
                    results.append({'id': bid, **it})
                break
            except Exception as e:
                wait = 60 if '429' in str(e) else 2 ** attempt
                print(f'  Retry {attempt+1}: {str(e)[:80]}')
                time.sleep(wait)
        time.sleep(sleep)
    return pd.DataFrame(results)

df_or = run_openrouter(cv_texts, cv_ids)
df_or = df_or.rename(columns={'label': 'label_or', 'conf': 'conf_or'})
print(f'OpenRouter: {len(df_or)} texts')
print(df_or['label_or'].value_counts().to_dict())


In [ ]:
# ── Merge cross-val results ───────────────────────────────────────────────
# Primary Groq labels become the reference model
cv = (sample[['id', 'label', 'conf']]
      .rename(columns={'label': 'label_groq', 'conf': 'conf_groq'})
      .merge(df_gemini[['id', 'label_gemini', 'conf_gemini']], on='id', how='inner')
      .merge(df_or[['id', 'label_or', 'conf_or']], on='id', how='inner'))

print(f'Cross-val merged: {len(cv)} texts with all 3 model labels')
cv.to_parquet(CROSSVAL_PATH, index=False)
print(f'Saved: {CROSSVAL_PATH}')
cv.head(5)


---
## F — Analysis & Diagnostics

Inter-model agreement is measured with **Cohen's \u03ba** (appropriate for categorical labels).  
We also plot confusion matrices and label distributions per model.

In [ ]:
# ── Cohen's Kappa pairwise agreement ─────────────────────────────────────
PAIRS = [
    ('label_groq',   'label_gemini', f'Groq vs Gemini'),
    ('label_groq',   'label_or',     f'Groq vs OpenRouter'),
    ('label_gemini', 'label_or',     f'Gemini vs OpenRouter'),
]

print('=' * 65)
print('INTER-MODEL AGREEMENT  (Cohen\u2019s \u03ba)')
print('=' * 65)
kappa_rows = []
for c1, c2, label in PAIRS:
    kappa   = cohen_kappa_score(cv[c1], cv[c2])
    pct_agr = (cv[c1] == cv[c2]).mean()
    if kappa >= 0.81:   verdict = '\u2713 Almost perfect'
    elif kappa >= 0.61: verdict = '~ Substantial'
    elif kappa >= 0.41: verdict = '~ Moderate'
    else:               verdict = '\u2717 Fair / Poor'
    print(f'  {label:<30}  \u03ba = {kappa:.3f}   %agree = {pct_agr:.1%}   {verdict}')
    kappa_rows.append({'pair': label, 'kappa': kappa, 'pct_agree': pct_agr})

kappa_df = pd.DataFrame(kappa_rows)
print()
print('\u03ba \u2265 0.81 = almost perfect  |  0.61\u20130.80 = substantial  |  '
      '0.41\u20130.60 = moderate  |  < 0.41 = low')


In [ ]:
# ── Visualisation: confusion matrices + label distributions ──────────────
CAT_COLORS = {'PRO_EQUALITY': '#2196F3', 'ANTI_EQUALITY': '#F44336', 'NEUTRAL': '#9E9E9E'}
MODEL_LABELS = {
    'label_groq':   'Groq\nLlama-3.3-70B',
    'label_gemini': f'Gemini\n{GEMINI_MODEL}',
    'label_or':     f'OpenRouter\n{OR_MODEL.split("/")[-1][:18]}',
}

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Equality Classifier \u2014 Cross-Validation Diagnostics', fontsize=14, fontweight='bold')

# Row 0: confusion matrices (row model = reference, col model = comparison)
for ax, (c1, c2, title) in zip(axes[0], PAIRS):
    cm = confusion_matrix(cv[c1], cv[c2], labels=CATEGORIES, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
                xticklabels=CATEGORIES, yticklabels=CATEGORIES,
                linewidths=0.5, annot_kws={'size': 9})
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Column model', fontsize=8)
    ax.set_ylabel('Row model', fontsize=8)
    ax.tick_params(axis='x', rotation=25, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

# Row 1: label distribution per model
for ax, col in zip(axes[1], ['label_groq', 'label_gemini', 'label_or']):
    counts = cv[col].value_counts().reindex(CATEGORIES, fill_value=0)
    bars = ax.bar(CATEGORIES, counts.values,
                  color=[CAT_COLORS[c] for c in CATEGORIES],
                  width=0.6, edgecolor='white')
    ax.bar_label(bars, labels=[f'{v/len(cv):.1%}' for v in counts.values], fontsize=9)
    ax.set_title(MODEL_LABELS[col], fontsize=10, fontweight='bold')
    ax.set_ylabel('Texts')
    ax.set_xticks(range(len(CATEGORIES)))
    ax.set_xticklabels(CATEGORIES, rotation=20, fontsize=8)
    ax.set_ylim(0, max(counts.values) * 1.18)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(CKPT_DIR + 'crossval_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to', CKPT_DIR)


In [ ]:
# ── Majority vote + disagreement analysis ────────────────────────────────
LABEL_COLS = ['label_groq', 'label_gemini', 'label_or']

def majority_vote(row):
    votes = [row[c] for c in LABEL_COLS]
    top, count = Counter(votes).most_common(1)[0]
    return pd.Series({'majority_label': top, 'consensus': count / 3})

cv[['majority_label', 'consensus']] = cv.apply(majority_vote, axis=1)

print('Majority-vote distribution:')
print(cv['majority_label'].value_counts().to_string())
print()
print(f'Full consensus (3/3):  {(cv["consensus"] == 1.0).mean():.1%}')
print(f'Majority (2+/3):       {(cv["consensus"] >= 2/3).mean():.1%}')
print()

# Disagreement deep-dive
discord = cv[cv['consensus'] < 1.0][LABEL_COLS + ['majority_label']]
print(f'Texts with any disagreement: {len(discord)} ({len(discord)/len(cv):.1%})')

# Most common disagreement patterns
discord['pattern'] = discord.apply(
    lambda r: tuple(sorted([r[c] for c in LABEL_COLS])), axis=1)
print('\nTop disagreement patterns:')
print(discord['pattern'].value_counts().head(10).to_string())


In [ ]:
# ── Final export ─────────────────────────────────────────────────────────
# Save updated cross-val with majority vote
cv.to_parquet(CROSSVAL_PATH, index=False)

# Primary results as final dataset (all 500k)
final = pd.read_parquet(PRIMARY_CKPT).rename(
    columns={'label': 'label_primary', 'conf': 'conf_primary'})
final.to_parquet(FINAL_PATH, index=False)

print(f'Primary results:  {len(final):,} texts  \u2192  {FINAL_PATH}')
print(f'Cross-val sample: {len(cv):,} texts  \u2192  {CROSSVAL_PATH}')
print()

# Summary table
summary = pd.DataFrame({
    'Model': ['Groq Llama-3.3-70B (primary)', f'Gemini {GEMINI_MODEL}', f'OpenRouter {OR_MODEL.split("/")[-1]}'],
    'Texts': [len(final), len(df_gemini), len(df_or)],
    'PRO_EQ %':  [final['label_primary'].eq('PRO_EQUALITY').mean(),
                  df_gemini['label_gemini'].eq('PRO_EQUALITY').mean(),
                  df_or['label_or'].eq('PRO_EQUALITY').mean()],
    'ANTI_EQ %': [final['label_primary'].eq('ANTI_EQUALITY').mean(),
                  df_gemini['label_gemini'].eq('ANTI_EQUALITY').mean(),
                  df_or['label_or'].eq('ANTI_EQUALITY').mean()],
    'NEUTRAL %': [final['label_primary'].eq('NEUTRAL').mean(),
                  df_gemini['label_gemini'].eq('NEUTRAL').mean(),
                  df_or['label_or'].eq('NEUTRAL').mean()],
})
print(summary.to_string(index=False))
